# Numerical CPP: structure / embedding / fused arms

`CPP.run_num` runs the **same** algorithm as `CPP.run`, but the per-residue values come from a pre-sliced numerical tensor (`dict_num_parts`) instead of an amino-acid→scale lookup. The key idea:

- **Your PLM embeddings ARE the `dict_num`.** A `{entry: (L, D)}` tensor (ProtT5, ESM, …) feeds straight in — there is no conversion step.
- `EmbeddingPreprocessor.build_scales` / `build_cat` only **name** the `D` dimensions (producing `df_scales` / `df_cat`); they are *not* a per-residue value source.
- The same entry point serves three arms — **embedding**, **structure-only**, and **fused** — differing only in which `dict_num` you pass.

In [1]:
import numpy as np
import aaanalysis as aa

aa.options["verbose"] = False
df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()

# Stand-in for a real PLM: a per-residue embedding tensor {entry: (L, D)}.
D = 8
rng = np.random.default_rng(0)
dict_emb = {e: rng.random((len(s), D)) for e, s in zip(df_seq["entry"], df_seq["sequence"])}
list(dict_emb.items())[0][0], list(dict_emb.values())[0].shape

('Q14802', (87, 8))

## Embedding arm

`build_scales` / `build_cat` turn the embedding corpus into `df_scales` / `df_cat` that **name** the `D` dimensions. `NumericalFeature.get_parts` slices both the sequence parts and the tensor with shared boundaries; `CPP.run_num` then consumes the tensor directly.

In [2]:
ep = aa.EmbeddingPreprocessor()
df_scales = ep.build_scales(df_seq=df_seq, dict_num=dict_emb)   # (20, D) — names the dims
df_cat = ep.build_cat(df_scales=df_scales)         # (D, 5) categories

nf = aa.NumericalFeature()
df_parts, dict_num_parts = nf.get_parts(df_seq=df_seq, dict_num=dict_emb)

cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales, df_cat=df_cat, verbose=False)
df_feat_emb = cpp.run_num(dict_num_parts=dict_num_parts, labels=labels, n_filter=10, n_jobs=1)
df_feat_emb.head()

/var/folders/sv/65tlch_10198qgmpwcp6408r0000gn/T/ipykernel_51120/183176525.py:2: UserWarning: Pseudo-scales are dataset-dependent (averaged over df_seq). For reproducible cross-dataset comparison, compute them once on a fixed reference corpus and reuse the resulting df_scales.
  df_scales = ep.build_scales(df_seq=df_seq, dict_num=dict_emb)   # (20, D) — names the dims


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD_C_JMD_C-Segment(3,4)-dim_0",Embeddings,Embeddings_cat0_subcat2,dim_0,,0.50,0.181,-0.181,0.057,0.077,0.000157,0.062193,"31,32,33,34,35"
1,"JMD_N_TMD_N-Pattern(C,6,9,13)-dim_4",Embeddings,Embeddings_cat1_subcat1,dim_4,,0.42,0.239,0.239,0.128,0.128,0.001499,0.074194,"8,12,15"
2,"TMD_C_JMD_C-Segment(10,14)-dim_7",Embeddings,Embeddings_cat2_subcat5,dim_7,,0.42,0.237,0.237,0.131,0.107,0.001499,0.084793,"33,34"
3,"TMD-Pattern(C,3,7,10,13)-dim_3",Embeddings,Embeddings_cat0_subcat3,dim_3,,0.42,0.155,0.155,0.093,0.065,0.001499,0.065950,"18,21,24,28"
4,"JMD_N_TMD_N-Segment(2,10)-dim_2",Embeddings,Embeddings_cat0_subcat0,dim_2,,0.40,0.304,0.304,0.190,0.140,0.002497,0.070627,"3,4"


## Fused arm

To combine sources (e.g. embeddings **and** structure features), concatenate the per-residue tensors along the `D` axis with `aa.combine_dict_nums` *first*, then run the identical pipeline. Here we add a 3-dim structure-like tensor to the 8-dim embedding for `D = 11`.

In [3]:
dict_struct = {e: rng.random((len(s), 3)) for e, s in zip(df_seq["entry"], df_seq["sequence"])}
dict_fused = aa.combine_dict_nums([dict_emb, dict_struct])   # D = 8 + 3 = 11

df_scales_f = ep.build_scales(df_seq=df_seq, dict_num=dict_fused)
df_cat_f = ep.build_cat(df_scales=df_scales_f)
df_parts_f, dnp_f = nf.get_parts(df_seq=df_seq, dict_num=dict_fused)

cpp_f = aa.CPP(df_parts=df_parts_f, df_scales=df_scales_f, df_cat=df_cat_f, verbose=False)
df_feat_fused = cpp_f.run_num(dict_num_parts=dnp_f, labels=labels, n_filter=10, n_jobs=1)
print("fused D =", df_scales_f.shape[1])
df_feat_fused.head()

/var/folders/sv/65tlch_10198qgmpwcp6408r0000gn/T/ipykernel_51120/2007555448.py:4: UserWarning: Pseudo-scales are dataset-dependent (averaged over df_seq). For reproducible cross-dataset comparison, compute them once on a fixed reference corpus and reuse the resulting df_scales.
  df_scales_f = ep.build_scales(df_seq=df_seq, dict_num=dict_fused)


fused D = 11


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD_C_JMD_C-Segment(3,4)-dim_0",Embeddings,Embeddings_cat0_subcat3,dim_0,,0.50,0.181,-0.181,0.057,0.077,0.000157,0.085436,"31,32,33,34,35"
1,"JMD_N_TMD_N-Pattern(C,6,9,13)-dim_4",Embeddings,Embeddings_cat1_subcat1,dim_4,,0.42,0.239,0.239,0.128,0.128,0.001499,0.090599,"8,12,15"
2,"TMD_C_JMD_C-Segment(10,14)-dim_7",Embeddings,Embeddings_cat2_subcat2,dim_7,,0.42,0.237,0.237,0.131,0.107,0.001499,0.101923,"33,34"
3,"TMD-Pattern(C,3,7,10,13)-dim_3",Embeddings,Embeddings_cat0_subcat0,dim_3,,0.42,0.155,0.155,0.093,0.065,0.001499,0.203847,"18,21,24,28"
4,"JMD_N_TMD_N-Segment(2,10)-dim_2",Embeddings,Embeddings_cat0_subcat4,dim_2,,0.40,0.304,0.304,0.190,0.140,0.002497,0.056597,"3,4"


**Structure-only** works the same way — pass a `dict_num` from `StructurePreprocessor` (DSSP / PDB / PAE encoders) instead of the embedding tensor. In every arm the only thing that changes is the `dict_num`; the `get_parts` → `run_num` pipeline and the `df_feat` output schema are identical to sequence-mode `CPP.run`.